# Multiclass · versi review
**Pilot satu kasus: EDA angka → preprocessing sesuai model → 5 pendekatan → evaluasi kelemahan.**

Run All pada **Kaggle GPU**, internet aktif. Semua angka EDA berasal dari data asli; tidak ada grafik otomatis atau hasil yang ditulis manual. Untuk dataset baru, ubah konfigurasi dan sumber input.

| Pendekatan | Representasi | Tujuan |
|---|---|---|
| Mayoritas | Tanpa fitur | Batas bawah |
| Logistic Regression | Word TF-IDF | Baseline teks; dua versi preprocessing diuji |
| Linear SVM | Word + char TF-IDF | Pola kata dan variasi ejaan |
| Logistic Regression | Mean Word2Vec, dilatih di train | Embedding statis |
| IndoBERTweet | Tokenizer + contextual BERT | Fine-tuning model bahasa |

**Data contoh:** [komentar YouTube MBG](https://www.kaggle.com/datasets/dzikriraihan/indonesian-youtube-comments-mbg-program). Label contoh diperlakukan sebagai pseudo-label mengikuti notebook awal. Confidence Score adalah keyakinan pemberi label, bukan bukti label benar. Skor terhadap label ini tidak membuktikan akurasi sentimen manusia.

**Validasi:** grup duplikat dibentuk sebelum split; train/validation/holdout sekitar 60/20/20. Semua model memakai baris yang sama. Validation untuk memilih varian/model dan epoch. Holdout hanya dievaluasi setelah keputusan dibekukan; skor validation adalah skor pengembangan, bukan estimasi final bebas bias.

## 00 · Konfigurasi

Fokus pilot 5.000 baris; N_ROWS=None untuk seluruh CSV. SOURCE_COL/GROUP_COL boleh None. GROUP_COL harus merepresentasikan unit yang tidak boleh bocor, misalnya video/thread/pengguna.

In [1]:
from pathlib import Path
SEED, N_ROWS, EDA_N, TOP = 42, 5000, 5000, 15
DATA_FILE = None  # None: cari comment_sentiment.csv di input Kaggle
TEXT_COL, LABEL_COL, CONF_COL = "Comment", "Sentiment", "Confidence Score"
SOURCE_COL, GROUP_COL = None, "Commenter ID"
MODEL_ID = "indolem/indobertweet-base-uncased"
EPOCHS, MAX_LEN, BATCH_SIZE = 3, 128, 16
NEAR_DUP, NEAR_THRESHOLD = True, .9
CLASS_WEIGHT, MIN_POST_GAIN = "balanced", .005
OUT = Path("/kaggle/working/hasil_review")

## 01 · Lingkungan Kaggle

Paket tambahan dipasang pada Kaggle. Tidak menginstal atau melatih model pada komputer lokal. Model pretrained memakai bundle Kaggle jika tersedia, atau Hugging Face bila tidak tersedia.

In [2]:
import sys,subprocess,os
assert Path("/kaggle/input").exists(), "Jalankan notebook ini di Kaggle."
subprocess.run([sys.executable,"-m","pip","install","-q",
    "gensim>=4.4,<5","ftfy>=6.3,<7","emoji>=2.14,<3","datasketch>=1.6,<3","langid==1.1.6"],check=True)
os.environ.update(TOKENIZERS_PARALLELISM="false",HF_HUB_DISABLE_PROGRESS_BARS="1",USE_TF="0",USE_FLAX="0")

In [3]:
import json,time,random,re,html,unicodedata,gc,hashlib
from collections import Counter
import numpy as np
import pandas as pd
import torch,emoji,ftfy
from IPython.display import display
from sklearn.metrics import f1_score,accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
assert torch.cuda.is_available(), "Aktifkan GPU Kaggle untuk menjalankan kelima pendekatan."
OUT.mkdir(exist_ok=True)
print("GPU:",torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
def show(title,table):
    print("\n"+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(3))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({"n":n,"%":n/(len(values) or np.nan)*100})

def macro(y,p):return f1_score(y,p,labels=np.arange(C),average="macro",zero_division=0)
pd.set_option("display.max_columns",None,"display.max_colwidth",130)
RUN_START=time.perf_counter()

## 02 · Muat data mentah & audit sebelum drop

Audit ini dilakukan sebelum membuang missing/teks invalid. Nama kolom tidak ditebak diam-diam. Informasi sumber/grup dipakai bila disediakan; tidak dibuat sintetis.

In [5]:
INPUT=Path("/kaggle/input")
path=Path(DATA_FILE) if DATA_FILE else next(INPUT.rglob("comment_sentiment.csv"))
raw=pd.read_csv(path)
assert {TEXT_COL,LABEL_COL}.issubset(raw), f"Kolom tersedia: {raw.columns.tolist()}"
show("Skema sebelum pembersihan",pd.DataFrame({"tipe":raw.dtypes.astype(str),"missing":raw.isna().sum(),
    "missing_%":raw.isna().mean()*100,"unik_non_null":raw.nunique(dropna=True)}))
print("File:",path,"| total baris:",len(raw),"| baris duplikat ekstra:",raw.duplicated().sum())


Skema sebelum pembersihan


,tipe,missing,missing_%,unik_non_null
Comment ID,object,0,0.0,5034
Comment Link,object,0,0.0,5034
Commenter Name,object,0,0.0,4431
Commenter ID,object,0,0.0,4431
Comment,object,0,0.0,4996
Likes,int64,0,0.0,45
Cleaned Comment,object,0,0.0,4996
Sentiment,object,0,0.0,3
Confidence Score,float64,0,0.0,4966


File: /kaggle/input/datasets/dzikriraihan/indonesian-youtube-comments-mbg-program/comment_sentiment.csv | total baris: 5035 | baris duplikat ekstra: 1


In [6]:
raw_text=raw[TEXT_COL].where(raw[TEXT_COL].map(lambda x:isinstance(x,str)),"").fillna("")
quality=pd.DataFrame({"teks_null":raw[TEXT_COL].isna(),"teks_non_string":raw[TEXT_COL].notna() & ~raw[TEXT_COL].map(lambda x:isinstance(x,str)),
    "teks_kosong":raw_text.str.strip().eq(""),"label_null":raw[LABEL_COL].isna(),
    "label_kosong":raw[LABEL_COL].astype("string").str.strip().eq("").fillna(False),
    "teks_pseudo_null":raw_text.str.strip().str.casefold().isin(["nan","null","none","n/a","-"])})
show("Kualitas input · flag dapat overlap",pd.DataFrame({"n":quality.sum(),"%":quality.mean()*100}))
valid=~quality.any(axis=1)
data=raw.loc[valid].copy().reset_index(names="row_id")
data[LABEL_COL]=data[LABEL_COL].astype(str).str.strip()
if N_ROWS and len(data)>N_ROWS:data=data.sample(N_ROWS,random_state=SEED).reset_index(drop=True)
print("Dikeluarkan karena invalid:",int((~valid).sum()),"| baris pilot:",len(data))


Kualitas input · flag dapat overlap


,n,%
teks_null,0,0.0
teks_non_string,0,0.0
teks_kosong,0,0.0
label_null,0,0.0
label_kosong,0,0.0
teks_pseudo_null,0,0.0


Dikeluarkan karena invalid: 0 | baris pilot: 5000


## 03 · Fungsi dari preprocessing universal
Fungsi berikut disalin langsung dari stepwise.py agar notebook tetap bisa dicopy sendiri. Pilihan sentimen: perbaikan teknis, placeholder sosial, normalisasi terkurasi, emoji tetap tersedia sebagai teks/karakter. Negasi, angka, dan istilah domain dipertahankan. Stemming, stopword removal, penghapusan emoji/angka, terjemahan daerah, dan typo otomatis tidak diaktifkan.

`minimal`: safe + casefold untuk TF-IDF/char. `normalized`: normalisasi terkurasi untuk eksperimen TF-IDF/Word2Vec. `bert`: safe + resep IndoBERTweet. Luasnya perubahan dibuktikan pada audit sebelum/sesudah, manfaatnya diuji pada validation.

In [7]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
STOPWORDS = set('yang dan di ke dari untuk pada dengan ini itu atau juga saya kami kita mereka dia oleh sebagai dalam para serta adalah sebuah'.split()) - NEGATIONS - PROTECTED
REGIONAL = {'ora': 'tidak', 'teu': 'tidak', 'indak': 'tidak', 'apik': 'bagus'}
OBFUSCATION = {'k3r4cunan': 'keracunan', 'k3r4cun4n': 'keracunan',
               'ker@cunan': 'keracunan', 'koru*si': 'korupsi'}
# URL/email/account/hashtag are single tokens: normalization never enters them.
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)


def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

In [8]:
def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

In [9]:
def decode_html_entities(text):
    return html.unescape(text)

def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [10]:
def replace_urls(text):
    # Preserve terminal sentence punctuation; a heuristic, not a complete URL parser.
    return URL.sub(lambda m: ' URL ' + m[0][len(m[0].rstrip('.,!?;)')):], text)

def replace_emails(text):
    return re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b', ' EMAIL ', text)

def replace_phones(text):
    return re.sub(r'(?<!\w)(?:\+62|62|0)[ -]?8(?:[ -]?\d){8,11}(?!\w)', ' PHONE ', text)

def replace_mentions(text):
    return re.sub(r'(?<![\w@])@[\w.]+', ' USER ', text)

In [11]:
def segment_hashtags(text, mapping=None):
    mapping = {} if mapping is None else mapping
    def split(match):
        word = match[1]
        # #tolakmbg stays tolakmbg unless an explicit reviewed map is supplied.
        return ' ' + mapping.get(word.casefold(), re.sub(
            r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word)) + ' '
    return re.sub(r'(?<!\w)#(\w+)', split, text)

def lowercase(text):
    return text.casefold()

def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

In [12]:
def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

def normalize_laughter(text):
    def replace(word):
        return 'wkwk' if re.fullmatch(r'(?:wk){2,}|w+k+w+k+[wk]*', word, re.I) else (
            'haha' if re.fullmatch(r'(?:ha){2,}h?', word, re.I) else word)
    return map_words(text, replace)

def normalize_reduplication(text, mapping=None):
    # A whitelist avoids turning grade S2, model GPT2, dates or quantities into words.
    mapping = {'anak2': 'anak-anak', 'orang2': 'orang-orang', 'teman2': 'teman-teman'} if mapping is None else mapping
    return re.sub(r'\b\w+2\b', lambda m: mapping.get(m[0].casefold(), m[0]), text)

In [13]:
def normalize_punctuation_repeat(text):
    return re.sub(r'([!?.,])\1{2,}', r'\1\1', text)

def describe_emoji(text):
    return emoji.demojize(text, delimiters=(' ', ' '))

def indobertweet_view(text):
    text = URL.sub('HTTPURL', text.lower())
    text = re.sub(r'(?<![\w@])@[\w.]+', '@USER', text)
    return normalize_whitespace(emoji.demojize(text))

In [14]:
def chain(text,steps):
    for step in steps:text=step(text)
    return text
safe_steps=[valid_text,repair_encoding,normalize_unicode,decode_html_entities,remove_invisible,normalize_whitespace]
normal_steps=[replace_emails,replace_urls,replace_phones,replace_mentions,segment_hashtags,lowercase,
    normalize_slang,normalize_elongation,normalize_laughter,normalize_reduplication,normalize_punctuation_repeat,describe_emoji,normalize_whitespace]
cache={s:chain(s,safe_steps) for s in data[TEXT_COL].unique()}
data["safe"]=data[TEXT_COL].map(cache)
print("Kosong setelah safe:",data.safe.eq("").sum())
data=data.loc[data.safe.ne("")].reset_index(drop=True)
data["minimal"]=data.safe.str.casefold()
cache={s:chain(s,normal_steps) for s in data.safe.unique()}
data["normalized"]=data.safe.map(cache)
data["bert"]=data.safe.map(indobertweet_view)

Kosong setelah safe: 0


## 04 · Grup duplikat sebelum split

Exact group memakai safe+casefold+spasi. Opsional near-duplicate: kandidat MinHash atas char 5-gram, diverifikasi Jaccard >= NEAR_THRESHOLD. Ini aproksimasi: kandidat yang terlewat tetap mungkin. Teks <20 karakter hanya exact; grup transitive. GROUP_COL yang diisi turut menyatukan grup.

In [15]:
def duplicate_groups(frame):
    from datasketch import MinHash,MinHashLSH
    keys=frame.safe.str.casefold().str.replace(r"\s+"," ",regex=True)
    parent=list(range(len(frame)))
    def root(i):
        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
        return i
    def merge(i,j):parent[root(i)]=root(j)
    first={}
    for i,key in enumerate(keys):
        if key in first:merge(i,first[key])
        else:first[key]=i
    if GROUP_COL:
        assert GROUP_COL in frame, f"Kolom grup tidak ada: {GROUP_COL}"
        for ids in frame.groupby(GROUP_COL).indices.values():
            for i in ids[1:]:merge(int(ids[0]),int(i))
    if NEAR_DUP:
        index=MinHashLSH(threshold=NEAR_THRESHOLD,num_perm=64)
        shingles={}
        for key,i in first.items():
            if len(key)<20:continue
            grams={key[j:j+5] for j in range(len(key)-4)}
            signature=MinHash(num_perm=64,seed=SEED)
            signature.update_batch([g.encode() for g in sorted(grams)])
            for candidate in index.query(signature):
                other=shingles[candidate]
                if len(grams&other)/len(grams|other)>=NEAR_THRESHOLD:merge(i,int(candidate))
            index.insert(str(i),signature);shingles[str(i)]=grams
    return np.array([root(i) for i in range(len(frame))])

In [16]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder().fit(data[LABEL_COL]);classes=encoder.classes_;C=len(classes)
assert C>=3, "Pilot ini untuk klasifikasi multiclass."
data["y"]=encoder.transform(data[LABEL_COL])
data["dup_group"]=duplicate_groups(data)
assert data.groupby("y").dup_group.nunique().min()>=5, "Perlu >=5 grup independen per kelas; ubah skema split untuk data kecil."
fold=np.full(len(data),-1)
for k,(_,idx) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(data,data.y,data.dup_group)):fold[idx]=k
train=data.loc[fold>=2].copy();val=data.loc[fold==1].copy();holdout=data.loc[fold==0].copy()
for a,b in [(train,val),(train,holdout),(val,holdout)]:assert set(a.dup_group).isdisjoint(b.dup_group)
for frame in [train,val,holdout]:assert set(frame.y)==set(range(C)), "Ada kelas hilang pada split; revisi skema sebelum training."
show("Pembagian baris",pd.DataFrame({name:frame[LABEL_COL].value_counts() for name,frame in [("train",train),("validation",val),("holdout",holdout)]}).fillna(0))
print("Grup independen:",data.dup_group.nunique(),"| overlap grup antar-split: 0")


Pembagian baris


,train,validation,holdout
Sentiment,,,
negative,2032,710,713
positive,680,228,216
neutral,254,78,89


Grup independen: 4381 | overlap grup antar-split: 0


## 05 · EDA wajib multiclass · angka

EDA mendalam berikut hanya train. Persentase noise memakai semua baris train; vocabulary memakai sampel train maksimal EDA_N. Validation/holdout tidak menjadi sumber kamus, vocabulary, Word2Vec, atau normalizer yang belajar dari data.

In [17]:
t=train[TEXT_COL].astype(str);words=t.str.split().str.len();chars=t.str.len()
key=train.safe.str.casefold().str.replace(r"\s+"," ",regex=True)
show("Distribusi kelas train",distribution(train[LABEL_COL]))
show("Panjang karakter/kata",pd.DataFrame({"karakter":chars,"kata":words}).describe(percentiles=[.01,.05,.5,.9,.95,.99,.999]).T)
show("Duplikat train",pd.Series({"persis_ekstra":t.duplicated().sum(),"normal_ekstra":key.duplicated().sum(),
    "normal_ekstra_%":key.duplicated().mean()*100,"grup_dup":key.value_counts().gt(1).sum()}))
show("Panjang per kelas",words.groupby(train[LABEL_COL]).describe(percentiles=[.5,.95,.99]))


Distribusi kelas train


,n,%
Sentiment,,
negative,2032,68.510
positive,680,22.927
neutral,254,8.564



Panjang karakter/kata


,count,mean,std,min,1%,5%,50%,90%,95%,99%,99.9%,max
karakter,2966.0,198.250,372.963,1.0,6.0,18.0,109.0,410.0,601.5,1369.3,4909.92,10262.0
kata,2966.0,29.422,53.048,1.0,1.0,3.0,16.5,60.0,87.0,204.7,651.21,1446.0



Duplikat train


,0
persis_ekstra,18.000
normal_ekstra,21.000
normal_ekstra_%,0.708
grup_dup,17.000



Panjang per kelas


,count,mean,std,min,50%,95%,99%,max
Sentiment,,,,,,,,
negative,2032.0,33.660,61.509,1.0,18.5,106.00,236.76,1446.0
neutral,254.0,10.343,8.936,1.0,8.0,25.70,43.35,74.0
positive,680.0,23.885,26.853,1.0,15.0,72.05,125.05,305.0


In [18]:
sample=t.sample(min(EDA_N,len(t)),random_state=SEED)
seqs=sample.str.casefold().str.findall(r"[^\W\d_]+")
counts=pd.Series(Counter(w for seq in seqs for w in seq),dtype="int64").sort_values(ascending=False)
docs=pd.Series(Counter(w for seq in seqs for w in set(seq)),dtype="int64")
show("Vocabulary train · sampel",pd.Series({"dokumen":len(seqs),"token":counts.sum(),"unik":len(counts),
    "TTR_%":100*len(counts)/(counts.sum() or np.nan),"hapax":counts.eq(1).sum(),"hapax_%_tipe":100*counts.eq(1).mean(),
    "hapax_%_token":100*counts.eq(1).sum()/(counts.sum() or np.nan),"langka_<=5_%_tipe":100*counts.le(5).mean()}))
show("Top words",counts.head(TOP).rename("n").to_frame().assign(pct_token=counts.head(TOP)/(counts.sum() or np.nan)*100,
    dokumen=docs.reindex(counts.head(TOP).index),pct_dokumen=docs.reindex(counts.head(TOP).index)/len(seqs)*100))


Vocabulary train · sampel


,0
dokumen,2966.000
token,91190.000
unik,10526.000
TTR_%,11.543
hapax,5648.000
hapax_%_tipe,53.658
hapax_%_token,6.194
langka_<=5_%_tipe,81.902



Top words


,n,pct_token,dokumen,pct_dokumen
mbg,2125,2.330,1320,44.504
br,1869,2.050,472,15.914
dan,1781,1.953,906,30.546
di,1753,1.922,941,31.726
yg,1693,1.857,855,28.827
yang,1177,1.291,564,19.016
ini,878,0.963,554,18.678
itu,842,0.923,565,19.049
ada,752,0.825,509,17.161
tidak,698,0.765,466,15.711


In [19]:
for n in [2,3]:
    bag=pd.Series(Counter(" ".join(seq[i:i+n]) for seq in seqs for i in range(len(seq)-n+1)),dtype="int64").sort_values(ascending=False)
    show(f"Top {n}-gram tanpa melompati stopword",bag.head(TOP).rename("n").to_frame().assign(pct=bag.head(TOP)/(bag.sum() or np.nan)*100))
show("Contoh hapax",counts[counts.eq(1)].head(TOP))
odd=counts.index.to_series().str.len().ge(20)|counts.index.to_series().str.count(r"([^\W\d_])\1{2,}").gt(0)
show("Kandidat kata panjang/berulang · bukan pasti typo",counts[odd].head(TOP))


Top 2-gram tanpa melompati stopword


,n,pct
br br,430,0.487
prof gema,135,0.153
program mbg,122,0.138
mbg itu,118,0.134
pak gema,97,0.110
mbg di,95,0.108
a href,87,0.099
lapangan kerja,79,0.090
mbg ini,77,0.087
lebih baik,75,0.085



Top 3-gram tanpa melompati stopword


,n,pct
www youtube com,71,0.083
https www youtube,52,0.061
youtube com watch,52,0.061
com watch v,52,0.061
a href https,51,0.060
href https www,51,0.060
watch v svi,50,0.059
v svi xhep,50,0.059
svi xhep j,50,0.059
xhep j amp,50,0.059



Contoh hapax


,0
mkaan,1
petarung,1
mma,1
ring,1
bersahabat,1
memukul,1
netapkan,1
menyahnyalahkan,1
cerna,1
tuwanya,1



Kandidat kata panjang/berulang · bukan pasti typo


,0
www,71
yaaa,6
dipertanggungjawabkan,4
yaaaa,4
nyeee,4
pppk,4
bosss,3
yesss,2
lahhh,2
hhhhh,2


In [20]:
NOISE={"url":r"https?://|www\.","mention":r"@\w+","hashtag":r"#\w+","angka":r"\d",
    "emoji/simbol":r"[\U0001F300-\U0001FAFF\u2600-\u27BF]","kapital":r"\b[A-Z]{3,}\b",
    "huruf_ulang":r"([^\W\d_])\1{2,}","baca_ulang":r"[!?.,]{2,}","html":r"</?[a-zA-Z][^>]*>",
    "encoding":r"\ufffd|Ã.|Â.","invisible":r"[\u200b\ufeff]","newline":r"[\n\r]",
    "email":r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}","non_ASCII":r"[^\x00-\x7f]",
    "html_entity":r"&(?:#\d+|[a-zA-Z]+);","spasi_tepi":r"^\s|\s$","spasi_ulang":r" {2,}",
    "kontrol":r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]","huruf_angka":r"\b(?=\w*[a-zA-Z])(?=\w*\d)\w+\b",
    "slang_contoh":r"\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b",
    "negasi":r"\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|gk)\b"}
flags=pd.DataFrame({name:t.str.count(pattern,flags=0 if name=="kapital" else re.I).gt(0) for name,pattern in NOISE.items()})
show("Noise train · % seluruh baris, flag overlap",pd.DataFrame({"n":flags.sum(),"%":flags.mean()*100}).sort_values("n",ascending=False))
show("Noise per kelas (%)",flags.groupby(train[LABEL_COL]).mean()*100)
if SOURCE_COL:
    assert SOURCE_COL in train
    show("Sumber",distribution(train[SOURCE_COL]));show("Label per sumber (%)",pd.crosstab(train[SOURCE_COL],train[LABEL_COL],normalize="index")*100)


Noise train · % seluruh baris, flag overlap


,n,%
kapital,1270,42.819
slang_contoh,1144,38.570
negasi,1088,36.682
angka,922,31.086
baca_ulang,820,27.647
huruf_angka,612,20.634
non_ASCII,557,18.780
html,526,17.734
emoji/simbol,487,16.419
spasi_ulang,473,15.947



Noise per kelas (%)


,url,mention,hashtag,angka,emoji/simbol,kapital,huruf_ulang,baca_ulang,html,encoding,invisible,newline,email,non_ASCII,html_entity,spasi_tepi,spasi_ulang,kontrol,huruf_angka,slang_contoh,negasi
Sentiment,,,,,,,,,,,,,,,,,,,,,
negative,1.772,0.098,1.575,34.252,14.272,46.014,6.053,30.020,18.504,0.0,0.000,0.246,0.0,16.535,8.809,0.0,17.766,0.0,23.376,43.701,44.439
neutral,6.693,0.394,2.362,22.047,11.811,35.827,9.055,10.630,12.205,0.0,0.000,0.000,0.0,12.992,8.661,0.0,7.480,0.0,13.386,14.567,8.661
positive,1.471,0.294,0.294,25.000,24.559,35.882,5.147,26.912,17.500,0.0,0.294,0.147,0.0,27.647,6.471,0.0,13.676,0.0,15.147,32.206,23.971


In [21]:
conflict=train.groupby(key)[LABEL_COL].nunique().loc[lambda s:s.gt(1)]
print("Grup teks dengan konflik label di train:",len(conflict))
show("Contoh konflik · perlu review konteks",train.loc[key.isin(conflict.index),[TEXT_COL,LABEL_COL]].head(TOP))
if CONF_COL in train:
    confidence=pd.to_numeric(train[CONF_COL],errors="coerce")
    show("Confidence pemberi label per kelas",confidence.groupby(train[LABEL_COL]).describe(percentiles=[.1,.5,.9]))
show("Contoh teks per kelas · train",train.groupby(LABEL_COL)[TEXT_COL].apply(lambda s:s.sample(min(3,len(s)),random_state=SEED).str.slice(0,180)).to_frame("teks"))

Grup teks dengan konflik label di train: 2

Contoh konflik · perlu review konteks


,Comment,Sentiment
474,setuju,negative
1886,B,negative
1950,b,neutral
3340,Setuju,positive



Confidence pemberi label per kelas


,count,mean,std,min,10%,50%,90%,max
Sentiment,,,,,,,,
negative,2032.0,0.921,0.132,0.401,0.709,0.988,0.999,1.000
neutral,254.0,0.770,0.183,0.382,0.508,0.806,0.984,0.999
positive,680.0,0.838,0.169,0.345,0.545,0.918,0.991,0.999



Contoh teks per kelas · train


teks
Sentiment                                                                                                                                        
negative  1700                                                              Kasian pak prabowo, niatnya baik tapi banyak penghianat disekitarnya😢
          2456                                                          Sayang pemerintah lambat mengepaluasi padahal sdh di kritik terus menerus
          1344                                                                                                                    Ini baru wow ..
neutral   3671                                                                                                                             Bazzer
          217                                <a href="https://www.youtube.com/watch?v=SVI8xheP-j4&amp;t=1824">30:24</a> jd sunk cost gak sih ?? 🤔
          1768  Daba Trilyunan .bila untuk buka lapangan kerja...spt cetak sawah.  buka perkebunan...yg tidak  gunduli hutan pegunungan ..beli...
positive  4692  Bila yg untuk sekolah tanpa sppg ,langsung siserahkan ke kantin sekolah anak &quot; bisa milih menu yg disuka jadi jelas kemak...
          4382  Mbg itu bagus sbnrnya uang negara balik ke rakyat lagi cuman emng harus diawasi dan diperketat lagi contoh nya untuk sekolah2 ...
          519   Syukur  anak Indonesia di kasih makan bergizi , lihat di  Palestina  anak anak makan aja susah , di Indonesia di kasih makan o...

In [22]:
from langid.langid import LanguageIdentifier,model as language_model
from threadpoolctl import threadpool_limits
language_sample=sample.sample(min(300,len(sample)),random_state=SEED)
identifier=LanguageIdentifier.from_modelstring(language_model,norm_probs=True)
with threadpool_limits(limits=1):
    language=language_sample.map(lambda s:identifier.classify(s[:3000]) if len(s.split())>=5 else ("tidak_pasti",np.nan))
lang=language.map(lambda p:p[0] if p[1]>=.8 else "tidak_pasti")
show("Estimasi bahasa · % sampel, satu bahasa dominan per dokumen",distribution(lang))
print("Bahasa serumpun/campur kode/teks pendek dapat salah; tidak_pasti bukan bahasa Indonesia otomatis.")


Estimasi bahasa · % sampel, satu bahasa dominan per dokumen


,n,%
Comment,,
id,157,52.333
tidak_pasti,107,35.667
ms,32,10.667
jv,2,0.667
nl,1,0.333
sw,1,0.333


Bahasa serumpun/campur kode/teks pendek dapat salah; tidak_pasti bukan bahasa Indonesia otomatis.


## 06 · Audit preprocessing & batas token

Fungsi deterministik boleh diterapkan pada semua split; semua statistik pemilihan berikut hanya train/validation. Negasi/emoji/angka tetap tersedia, meskipun normalisasi dapat mengubah bentuknya. Tidak membuang baris valid berdasarkan distribusi label.

In [23]:
audit=[]
for view in ["minimal","normalized","bert"]:
    audit.append({"versi":view,"berubah_%":train[view].ne(train.safe).mean()*100,"kosong":train[view].eq("").sum(),
        "unik":train[view].nunique(),"kata_p50":train[view].str.split().str.len().median(),"kata_p95":train[view].str.split().str.len().quantile(.95)})
show("Dampak preprocessing train",pd.DataFrame(audit).set_index("versi"))
show("Before/after train",train[[TEXT_COL,"minimal","normalized","bert"]].head(5))
assert all(frame[["minimal","normalized","bert"]].ne("").all().all() for frame in [train,val,holdout])


Dampak preprocessing train


,berubah_%,kosong,unik,kata_p50,kata_p95
versi,,,,,
minimal,92.009,0,2945,16.5,87.00
normalized,96.797,0,2945,17.0,88.75
bert,93.729,0,2945,16.5,87.00



Before/after train


,Comment,minimal,normalized,bert
0,❤❤,❤❤,red_heart red_heart,:red_heart::red_heart:
1,Analisis paling guooblook krn mengabaikan norma dan tata nilai,analisis paling guooblook krn mengabaikan norma dan tata nilai,analisis paling guooblook karena mengabaikan norma dan tata nilai,analisis paling guooblook krn mengabaikan norma dan tata nilai
3,"benar kemungkinan supplier nakal itu besar, tapi kenapa kenakalan dari purchasing dapurnya sendiri tidak di singgung? banyak l...","benar kemungkinan supplier nakal itu besar, tapi kenapa kenakalan dari purchasing dapurnya sendiri tidak di singgung? banyak l...","benar kemungkinan supplier nakal itu besar, tapi kenapa kenakalan dari purchasing dapurnya sendiri tidak di singgung? banyak l...","benar kemungkinan supplier nakal itu besar, tapi kenapa kenakalan dari purchasing dapurnya sendiri tidak di singgung? banyak l..."
7,"Master Gema saya sll menonton hampir semua podcast Master Gema baik ttf pasar modal ataupun ekonomi NKRI, saya setuju kita me...","master gema saya sll menonton hampir semua podcast master gema baik ttf pasar modal ataupun ekonomi nkri, saya setuju kita men...","master gema saya sll menonton hampir semua podcast master gema baik ttf pasar modal ataupun ekonomi nkri, saya setuju kita men...","master gema saya sll menonton hampir semua podcast master gema baik ttf pasar modal ataupun ekonomi nkri, saya setuju kita men..."
9,MURIT ATAU SISWA ITU DALAM SEKOLAH TIDAK DI BEDA2 LAN<br>BAGAIMANA CARA MEMBAGIKAN KE SISWA<br>MOHON PENJELASAN,murit atau siswa itu dalam sekolah tidak di beda2 lan<br>bagaimana cara membagikan ke siswa<br>mohon penjelasan,murit atau siswa itu dalam sekolah tidak di beda2 lan<br>bagaimana cara membagikan ke siswa<br>mohon penjelasan,murit atau siswa itu dalam sekolah tidak di beda2 lan<br>bagaimana cara membagikan ke siswa<br>mohon penjelasan


In [24]:
from transformers import AutoTokenizer
candidates=list(INPUT.glob("**/models/indobertweet/config.json"))
model_path=str(candidates[0].parent) if candidates else MODEL_ID
tokenizer=AutoTokenizer.from_pretrained(model_path,local_files_only=bool(candidates))
token_lengths=train.bert.map(lambda s:len(tokenizer.encode(s,truncation=False)))
show("Panjang token model train",token_lengths.describe(percentiles=[.5,.9,.95,.99]))
show("Truncation train · % baris tiap kelas",pd.DataFrame({str(k):token_lengths.gt(k) for k in [64,128,256,512]}).groupby(train[LABEL_COL]).mean()*100)
print("Batas run:",MAX_LEN,"| terpotong:",f"{token_lengths.gt(MAX_LEN).mean():.2%}")


Panjang token model train


,bert
count,2966.000
mean,46.494
std,80.608
min,3.000
50%,27.000
90%,91.500
95%,139.750
99%,314.000
max,2309.000



Truncation train · % baris tiap kelas


,64,128,256,512
Sentiment,,,,
negative,20.325,7.431,2.165,0.394
neutral,2.756,0.394,0.000,0.000
positive,16.029,3.971,0.294,0.000


Batas run: 128 | terpotong: 6.04%


## 07 · Pencatat evaluasi validation

Semua pendekatan memakai train/validation yang sama. Macro-F1 dipilih sebagai metrik kasus ini; accuracy/weighted-F1/balanced accuracy ikut dilaporkan. TF-IDF, Word2Vec, bobot kelas hanya belajar dari train. Holdout belum diprediksi.

In [25]:
from sklearn.pipeline import make_pipeline,make_union
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import FunctionTransformer
from gensim.models import Word2Vec
models={};val_pred={};val_prob={};records=[]
y_train,y_val=train.y.to_numpy(),val.y.to_numpy()
def metrics(y,p):
    return {"macro_f1":macro(y,p),"weighted_f1":f1_score(y,p,average="weighted",zero_division=0),
        "accuracy":accuracy_score(y,p),"balanced_acc":balanced_accuracy_score(y,p)}

def register(name,estimator,view):
    start=time.perf_counter();estimator.fit(train[view],y_train)
    seconds=time.perf_counter()-start
    models[name]=(estimator,view);val_pred[name]=estimator.predict(val[view])
    if hasattr(estimator,"predict_proba"):val_prob[name]=estimator.predict_proba(val[view])
    records.append({"model":name,"view":view,"fit_seconds":seconds,**metrics(y_val,val_pred[name])})
    print(name,records[-1])

## 08 · Model 1: mayoritas

Sanity baseline. Hanya memakai distribusi label train.

In [26]:
register("01_mayoritas",DummyClassifier(strategy="most_frequent"),"minimal")

01_mayoritas {'model': '01_mayoritas', 'view': 'minimal', 'fit_seconds': 0.0017010160000268115, 'macro_f1': 0.2742371572035535, 'weighted_f1': 0.57492632366493, 'accuracy': 0.6988188976377953, 'balanced_acc': np.float64(0.3333333333333333)}


## 09 · Model 2: TF-IDF word + Logistic Regression

Uji minimal vs normalized pada validation. Dua varian dihitung sebagai satu keluarga model. Pemilihan preprocessing pada validation membuat skor validation optimistis; holdout tetap terpisah.

In [27]:
def word_tfidf():
    return TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=1.,max_features=60000,sublinear_tf=True)
def logistic():return LogisticRegression(C=2,max_iter=1000,class_weight=CLASS_WEIGHT,random_state=SEED)
variants={};ablation=[]
for view in ["minimal","normalized"]:
    estimator=make_pipeline(word_tfidf(),logistic())
    start=time.perf_counter();estimator.fit(train[view],y_train)
    variants[view]=estimator
    ablation.append({"view":view,"fit_seconds":time.perf_counter()-start,**metrics(y_val,estimator.predict(val[view]))})
show("Ablasi preprocessing · validation",pd.DataFrame(ablation))
best_view=max(ablation,key=lambda r:r["macro_f1"])["view"]
name="02_word_tfidf_lr";models[name]=(variants[best_view],best_view)
val_pred[name]=variants[best_view].predict(val[best_view]);val_prob[name]=variants[best_view].predict_proba(val[best_view])
records.append({"model":name,**next(r for r in ablation if r["view"]==best_view)})


Ablasi preprocessing · validation


,view,fit_seconds,macro_f1,weighted_f1,accuracy,balanced_acc
0,minimal,2.589,0.606,0.773,0.780,0.595
1,normalized,1.182,0.606,0.775,0.781,0.598


## 10 · Model 3: word+char TF-IDF + Linear SVM

Char n-gram membantu menangkap variasi ejaan dan fragmen kata; manfaat tetap dinilai dari validation. SVM menghasilkan margin, bukan probabilitas terkalibrasi.

In [28]:
char_tfidf=TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=80000,sublinear_tf=True)
svm=make_pipeline(make_union(word_tfidf(),char_tfidf),LinearSVC(C=1,class_weight=CLASS_WEIGHT,random_state=SEED))
register("03_word_char_svm",svm,"minimal")

03_word_char_svm {'model': '03_word_char_svm', 'view': 'minimal', 'fit_seconds': 2.0358950440000285, 'macro_f1': 0.5530204014709671, 'weighted_f1': 0.7543110941678282, 'accuracy': 0.7726377952755905, 'balanced_acc': np.float64(0.5366029284116757)}


## 11 · Model 4: Word2Vec + Logistic Regression

Word2Vec dilatih hanya pada train, bukan mengambil embedding dari validation/holdout. Mean pooling menghilangkan urutan kata; embedding korpus kecil mungkin lemah. Coverage/OOV validation dan vektor nol dilaporkan.

In [29]:
def w2v_tokens(text):return re.findall(r"[^\W\d_]+",text.casefold())
train_tokens=train.normalized.map(w2v_tokens)
w2v_start=time.perf_counter()
w2v=Word2Vec(sentences=train_tokens,vector_size=100,window=5,min_count=2,sg=1,epochs=20,workers=1,seed=SEED,
    hashfxn=lambda s:int(hashlib.md5(s.encode()).hexdigest()[:8],16))

def mean_w2v(texts):
    rows=[]
    for text in texts:
        vectors=[w2v.wv[w] for w in w2v_tokens(text) if w in w2v.wv]
        rows.append(np.mean(vectors,axis=0) if vectors else np.zeros(w2v.vector_size))
    return np.asarray(rows,dtype="float32")
register("04_word2vec_lr",make_pipeline(FunctionTransformer(mean_w2v),logistic()),"normalized")
records[-1]["fit_seconds"]=time.perf_counter()-w2v_start

04_word2vec_lr {'model': '04_word2vec_lr', 'view': 'normalized', 'fit_seconds': 0.3925653969999985, 'macro_f1': 0.5523300427697063, 'weighted_f1': 0.6885722359101949, 'accuracy': 0.6633858267716536, 'balanced_acc': np.float64(0.6070489061222938)}


In [30]:
val_tokens=val.normalized.map(w2v_tokens)
known=sum(w in w2v.wv for seq in val_tokens for w in seq);total=sum(map(len,val_tokens))
show("Coverage Word2Vec validation",pd.Series({"vocab_train":len(w2v.wv),"token_val":total,"OOV_%":100*(1-known/(total or np.nan)),
    "dokumen_vektor_nol_%":100*np.all(mean_w2v(val.normalized)==0,axis=1).mean()}))
word_vocab=models["02_word_tfidf_lr"][0][0].vocabulary_
show("OOV word TF-IDF validation",pd.Series({"vocab_train":len(word_vocab),
    "catatan_n":sum(len(seq) for seq in val_tokens),"token_di_luar_vocab_%":100*sum(w not in word_vocab for seq in val_tokens for w in seq)/(total or np.nan)}))


Coverage Word2Vec validation


,0
vocab_train,4905.000
token_val,30772.000
OOV_%,9.665
dokumen_vektor_nol_%,1.083



OOV word TF-IDF validation


,0
vocab_train,13462.000
catatan_n,30772.000
token_di_luar_vocab_%,14.211


## 12 · Model 5: fine-tuning IndoBERTweet

Trainer memilih checkpoint dari validation macro-F1. Loss berbobot kelas dihitung dari train. Dynamic padding + truncation MAX_LEN. Recipe tokenizer mengikuti model card; tidak menambahkan stemming/slang normalization pada BERT.

In [31]:
from transformers import AutoModelForSequenceClassification,DataCollatorWithPadding,Trainer,TrainingArguments,set_seed
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
set_seed(SEED)
def encode(frame):
    dataset=Dataset.from_dict({"text":frame.bert.tolist(),"labels":frame.y.tolist()})
    return dataset.map(lambda batch:tokenizer(batch["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns="text")
encoded_train,encoded_val=encode(train),encode(val)
weights=compute_class_weight("balanced",classes=np.arange(C),y=y_train) if CLASS_WEIGHT=="balanced" else np.ones(C)
class WeightedTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop("labels");output=model(**inputs)
        loss=torch.nn.functional.cross_entropy(output.logits,labels,weight=torch.tensor(weights,device=output.logits.device,dtype=output.logits.dtype))
        return (loss,output) if return_outputs else loss

Map:   0%|          | 0/2966 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

In [32]:
import inspect
argument_names=inspect.signature(TrainingArguments).parameters
args=dict(output_dir=str(OUT/"bert_checkpoints"),num_train_epochs=EPOCHS,learning_rate=2e-5,weight_decay=.01,
    per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=BATCH_SIZE*2,save_strategy="epoch",logging_strategy="epoch",
    load_best_model_at_end=True,metric_for_best_model="macro_f1",greater_is_better=True,save_total_limit=1,
    report_to="none",disable_tqdm=True,seed=SEED,fp16=True,dataloader_num_workers=0)
args["eval_strategy" if "eval_strategy" in argument_names else "evaluation_strategy"]="epoch"
model=AutoModelForSequenceClassification.from_pretrained(model_path,num_labels=C,
    id2label=dict(enumerate(classes)),label2id={v:i for i,v in enumerate(classes)},ignore_mismatched_sizes=True)
trainer=WeightedTrainer(model=model,args=TrainingArguments(**args),train_dataset=encoded_train,eval_dataset=encoded_val,
    data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=lambda p:{"macro_f1":macro(p.label_ids,p.predictions.argmax(1))})
start=time.perf_counter();trainer.train();bert_seconds=time.perf_counter()-start

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[transformers] BertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline/models/indobertweet
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. 

{'loss': '0.8843', 'grad_norm': '8.806', 'learning_rate': '1.341e-05', 'epoch': '1'}


{'eval_loss': '0.6578', 'eval_macro_f1': '0.6641', 'eval_runtime': '3.89', 'eval_samples_per_second': '261.2', 'eval_steps_per_second': '4.113', 'epoch': '1'}


{'loss': '0.5338', 'grad_norm': '10.86', 'learning_rate': '6.738e-06', 'epoch': '2'}


{'eval_loss': '0.5448', 'eval_macro_f1': '0.683', 'eval_runtime': '4.137', 'eval_samples_per_second': '245.6', 'eval_steps_per_second': '3.868', 'epoch': '2'}


{'loss': '0.3953', 'grad_norm': '6.612', 'learning_rate': '7.168e-08', 'epoch': '3'}


{'eval_loss': '0.5373', 'eval_macro_f1': '0.7213', 'eval_runtime': '4.389', 'eval_samples_per_second': '231.5', 'eval_steps_per_second': '3.645', 'epoch': '3'}


{'train_runtime': '135.8', 'train_samples_per_second': '65.5', 'train_steps_per_second': '2.054', 'train_loss': '0.6045', 'epoch': '3'}


In [33]:
from scipy.special import softmax
logits=trainer.predict(encoded_val).predictions
val_prob["05_indobertweet"]=softmax(logits,axis=1)
val_pred["05_indobertweet"]=logits.argmax(1)
records.append({"model":"05_indobertweet","view":"bert","fit_seconds":bert_seconds,**metrics(y_val,logits.argmax(1))})
show("Riwayat training BERT",pd.DataFrame(trainer.state.log_history))
show("Perbandingan validation",pd.DataFrame(records).sort_values("macro_f1",ascending=False))


Riwayat training BERT


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_macro_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,0.884,8.806,0.0,1.0,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,93,0.658,0.664,3.890,261.205,4.113,NaN,NaN,NaN,NaN,NaN
2,0.534,10.858,0.0,2.0,186,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,186,0.545,0.683,4.137,245.594,3.868,NaN,NaN,NaN,NaN,NaN
4,0.395,6.612,0.0,3.0,279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,3.0,279,0.537,0.721,4.389,231.466,3.645,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,3.0,279,NaN,NaN,NaN,NaN,NaN,135.847,65.5,2.054,5.683373e+14,0.604



Perbandingan validation


,model,view,fit_seconds,macro_f1,weighted_f1,accuracy,balanced_acc
4,05_indobertweet,bert,136.474,0.721,0.811,0.803,0.773
1,02_word_tfidf_lr,normalized,1.182,0.606,0.775,0.781,0.598
2,03_word_char_svm,minimal,2.036,0.553,0.754,0.773,0.537
3,04_word2vec_lr,normalized,10.378,0.552,0.689,0.663,0.607
0,01_mayoritas,minimal,0.002,0.274,0.575,0.699,0.333


## 13 · Pilih model & post-processing

Model dipilih dari validation macro-F1. Pengali kelas hanya dicoba bila pemenang punya predict_proba. Dampak pengali diuji cross-fit dua fold grup pada validation; angka ini tetap skor pengembangan karena model juga dipilih di validation. Holdout belum dibuka. SVM margin tidak diperlakukan sebagai probabilitas.

In [34]:
leaderboard=pd.DataFrame(records).sort_values("macro_f1",ascending=False).reset_index(drop=True)
best_name=leaderboard.loc[0,"model"]
print("Pemenang validation:",best_name)
post_weights=np.ones(C);use_post=False

def fit_multipliers(prob,target):
    weights=np.ones(C)
    for _ in range(2):
        for k in range(C):
            best=macro(target,(prob*weights).argmax(1));chosen=weights[k]
            for value in [.5,.75,1.,1.25,1.5,2.]:
                candidate=weights.copy();candidate[k]=value
                score=macro(target,(prob*candidate).argmax(1))
                if score>best+1e-10:best,chosen=score,value
            weights[k]=chosen
    return weights

Pemenang validation: 05_indobertweet


In [35]:
if best_name in val_prob:
    prob=val_prob[best_name];crossfit=np.empty(len(val),dtype=int)
    for tune,check in StratifiedGroupKFold(2,shuffle=True,random_state=SEED).split(val,y_val,val.dup_group):
        weights=fit_multipliers(prob[tune],y_val[tune])
        crossfit[check]=(prob[check]*weights).argmax(1)
    before,after=macro(y_val,prob.argmax(1)),macro(y_val,crossfit)
    use_post=bool(after>before+MIN_POST_GAIN)
    if use_post:post_weights=fit_multipliers(prob,y_val)
    show("Pengali kelas · validation cross-fit",pd.Series({"raw_macro_f1":before,"crossfit_macro_f1":after,"dipakai":use_post}))
else: print("Post-processing dilewati: pemenang SVM menghasilkan margin, bukan probabilitas.")
selection={"best_validation_model":best_name,"postprocessing":use_post,"multipliers":post_weights.tolist(),"classes":classes.tolist()}
(OUT/"selection.json").write_text(json.dumps(selection,indent=2))
print("Keputusan dibekukan sebelum holdout:",selection)


Pengali kelas · validation cross-fit


,0
raw_macro_f1,0.721332
crossfit_macro_f1,0.741618
dipakai,True


Keputusan dibekukan sebelum holdout: {'best_validation_model': '05_indobertweet', 'postprocessing': True, 'multipliers': [1.5, 1.0, 0.5], 'classes': ['negative', 'neutral', 'positive']}


## 14 · Evaluasi holdout sekali

Semua model kini dievaluasi pada holdout yang sama. Pemilihan model/pengali tidak diubah berdasarkan tabel holdout. Waktu inferensi dihitung per 1.000 teks, tidak termasuk fit. Skor terhadap pseudo-label bukan skor terhadap anotasi manusia.

In [36]:
y_hold=holdout.y.to_numpy();hold_pred={};hold_prob={};hold_scores={};test_records=[]
for name,(estimator,view) in models.items():
    start=time.perf_counter();hold_pred[name]=estimator.predict(holdout[view])
    seconds=time.perf_counter()-start
    if hasattr(estimator,"predict_proba"):hold_prob[name]=estimator.predict_proba(holdout[view])
    elif hasattr(estimator,"decision_function"):hold_scores[name]=estimator.decision_function(holdout[view])
    test_records.append({"model":name,"infer_seconds_per1000":seconds/len(holdout)*1000,**metrics(y_hold,hold_pred[name])})
start=time.perf_counter();bert_logits=trainer.predict(encode(holdout)).predictions
hold_prob["05_indobertweet"]=softmax(bert_logits,axis=1);hold_pred["05_indobertweet"]=bert_logits.argmax(1)
test_records.append({"model":"05_indobertweet","infer_seconds_per1000":(time.perf_counter()-start)/len(holdout)*1000,**metrics(y_hold,bert_logits.argmax(1))})
assert len(hold_pred)==5 and all(len(pred)==len(holdout) for pred in hold_pred.values())
hold_table=pd.DataFrame(test_records).set_index("model")
show("Hasil holdout · urutan model validation",hold_table.reindex(leaderboard.model))

Map:   0%|          | 0/1018 [00:00<?, ? examples/s]


Hasil holdout · urutan model validation


,infer_seconds_per1000,macro_f1,weighted_f1,accuracy,balanced_acc
model,,,,,
05_indobertweet,4.431,0.709,0.797,0.786,0.765
02_word_tfidf_lr,0.049,0.604,0.741,0.739,0.611
03_word_char_svm,0.381,0.562,0.737,0.754,0.541
04_word2vec_lr,0.079,0.569,0.694,0.671,0.631
01_mayoritas,0.001,0.275,0.577,0.700,0.333


In [37]:
final_pred=(hold_prob[best_name]*post_weights).argmax(1) if use_post else hold_pred[best_name]
show("Pilihan final · holdout",pd.DataFrame([{"model":best_name,"postprocessing":use_post,**metrics(y_hold,final_pred)}]))
per_class=[]
for name,pred in {**hold_pred,"final_selected":final_pred}.items():
    report=classification_report(y_hold,pred,labels=np.arange(C),target_names=classes,output_dict=True,zero_division=0)
    per_class.extend({"model":name,"kelas":label,**report[label]} for label in classes)
per_class=pd.DataFrame(per_class)
show("Precision/recall/F1/support semua model",per_class)
cm=pd.DataFrame(confusion_matrix(y_hold,final_pred,labels=np.arange(C)),index=classes,columns=classes)
show("Confusion final · baris=aktual, kolom=prediksi",cm)
show("Confusion final · % per kelas aktual",cm.div(cm.sum(axis=1),axis=0)*100)


Pilihan final · holdout


,model,postprocessing,macro_f1,weighted_f1,accuracy,balanced_acc
0,05_indobertweet,True,0.707,0.811,0.805,0.742



Precision/recall/F1/support semua model


,model,kelas,precision,recall,f1-score,support
0,01_mayoritas,negative,0.700,1.000,0.824,713.0
1,01_mayoritas,neutral,0.000,0.000,0.000,89.0
2,01_mayoritas,positive,0.000,0.000,0.000,216.0
3,02_word_tfidf_lr,negative,0.842,0.823,0.833,713.0
4,02_word_tfidf_lr,neutral,0.385,0.416,0.400,89.0
5,02_word_tfidf_lr,positive,0.569,0.593,0.580,216.0
6,03_word_char_svm,negative,0.804,0.892,0.846,713.0
7,03_word_char_svm,neutral,0.400,0.202,0.269,89.0
8,03_word_char_svm,positive,0.626,0.528,0.573,216.0
9,04_word2vec_lr,negative,0.867,0.704,0.777,713.0



Confusion final · baris=aktual, kolom=prediksi


,negative,neutral,positive
negative,610,46,57
neutral,14,61,14
positive,46,22,148



Confusion final · % per kelas aktual


,negative,neutral,positive
negative,85.554,6.452,7.994
neutral,15.730,68.539,15.730
positive,21.296,10.185,68.519


## 15 · Di mana model lemah?

Slice menampilkan n, error rate, accuracy, dan macro-F1 untuk kelas aktual yang hadir dalam slice. F1 slice dengan komposisi kelas berbeda tidak langsung sebanding; lihat support/error rate. Confidence label rendah tidak otomatis berarti label salah.

In [38]:
analysis=holdout.copy()
analysis["prediksi"]=encoder.inverse_transform(final_pred)
analysis["salah"]=final_pred!=y_hold
analysis["panjang"]=pd.cut(analysis.safe.str.split().str.len(),[-1,5,15,40,100,np.inf])
for name in ["negasi","slang_contoh","url","emoji/simbol"]:
    analysis[name]=analysis[TEXT_COL].astype(str).str.count(NOISE[name],flags=re.I).gt(0)
analysis["dipotong_token"]=analysis.bert.map(lambda s:len(tokenizer.encode(s,truncation=False))>MAX_LEN)
if CONF_COL in analysis:analysis["confidence_label"]=pd.cut(pd.to_numeric(analysis[CONF_COL],errors="coerce"),[-np.inf,.6,.8,.95,np.inf])
slice_cols=["panjang","negasi","slang_contoh","url","emoji/simbol","dipotong_token"]
if CONF_COL in analysis:slice_cols.append("confidence_label")
if SOURCE_COL:slice_cols.append(SOURCE_COL)

In [39]:
slice_rows=[]
for column in slice_cols:
    for value,frame in analysis.groupby(column,observed=True,dropna=False):
        target=frame.y.to_numpy();pred=encoder.transform(frame.prediksi)
        slice_rows.append({"slice":column,"nilai":str(value),"n":len(frame),"error_%":frame.salah.mean()*100,
            "accuracy":accuracy_score(target,pred),"macro_f1_kelas_hadir":f1_score(target,pred,labels=np.unique(target),average="macro",zero_division=0)})
slices=pd.DataFrame(slice_rows)
show("Performa menurut kondisi teks",slices)
if best_name in hold_prob:
    analysis["skor_model"]=hold_prob[best_name].max(axis=1);score_kind="probabilitas_raw_belum_terkalibrasi"
else:
    margins=np.sort(hold_scores[best_name],axis=1)
    analysis["skor_model"]=margins[:,-1]-margins[:,-2];score_kind="selisih_margin_SVM"
columns=[TEXT_COL,LABEL_COL,"prediksi","skor_model"]+([CONF_COL] if CONF_COL in analysis else [])
print("Jenis skor:",score_kind,"| jika pengali aktif, skor di sini tetap skor model sebelum pengali.")
show("Salah prediksi paling yakin · butuh review manusia",analysis[analysis.salah].sort_values("skor_model",ascending=False)[columns].head(TOP))


Performa menurut kondisi teks


,slice,nilai,n,error_%,accuracy,macro_f1_kelas_hadir
0,panjang,"(-1.0, 5.0]",145,33.793,0.662,0.626
1,panjang,"(5.0, 15.0]",304,24.671,0.753,0.695
2,panjang,"(15.0, 40.0]",380,13.947,0.861,0.727
3,panjang,"(40.0, 100.0]",155,12.903,0.871,0.526
4,panjang,"(100.0, inf]",34,5.882,0.941,0.883
5,negasi,False,632,24.209,0.758,0.700
6,negasi,True,386,11.917,0.881,0.688
7,slang_contoh,False,619,23.425,0.766,0.699
8,slang_contoh,True,399,13.534,0.865,0.708
9,url,False,992,19.052,0.809,0.710


Jenis skor: probabilitas_raw_belum_terkalibrasi | jika pengali aktif, skor di sini tetap skor model sebelum pengali.

Salah prediksi paling yakin · butuh review manusia


,Comment,Sentiment,prediksi,skor_model,Confidence Score
1712,"MANTAB,,PROF GEMA INI PASTI CERDAS DAN IQNYA TINGGI NUTRISI PROTEINNYA PASTI LANCAR KE OTAK DARI BAYI SAMPE DEWASA,,KALO 25 % ...",negative,positive,0.977,0.659
2890,inti permaslahanya adalah bukan program.semuanya adalah soal tata kelola dan orang yag dipilih inkompeten.sebelum program ini ...,positive,negative,0.976,0.743
4724,Mantaf pencerahannya....,negative,positive,0.973,0.795
1444,Next bahas KOPDES prof,negative,neutral,0.970,0.541
144,"Mantab,, setuju sekaleee",negative,positive,0.968,0.836
4234,Setuju Mr PROF GEMA BETUL BNGET,negative,positive,0.967,0.987
3664,"Inti nya tetap unfaedah, tikus nya banyak.",positive,negative,0.957,0.739
568,Good job pak.<br>Pinter menjelaskan,negative,positive,0.956,0.500
149,SALUT.... ❤❤❤❤,negative,positive,0.951,0.968
4656,MBG lebih bijak dan lebih baik Ckup hanya sampai SD Saja. Karena di usia segitu otak dan tubuh memang sangat membutuhkan Nutri...,negative,positive,0.947,0.768


In [40]:
word_model=models["02_word_tfidf_lr"][0]
features=word_model[0].get_feature_names_out();coefficients=word_model[-1].coef_
show("Kata/ngram pendorong tiap kelas · asosiasi model linear",pd.DataFrame({label:features[np.argsort(coefficients[i])[-10:][::-1]] for i,label in enumerate(classes)}))
weakest=per_class[per_class.model.eq("final_selected")].sort_values("f1-score").iloc[0]
print("Kelas F1 terendah:",weakest.kelas,"| F1:",round(weakest["f1-score"],3),"| support:",int(weakest.support))
print("Slice error tertinggi dengan n>=20:")
display(slices[slices.n.ge(20)].sort_values("error_%",ascending=False).head(5))


Kata/ngram pendorong tiap kelas · asosiasi model linear


,negative,neutral,positive
0,tidak,menteri,mantap
1,korupsi,program untuk,banget
2,karena,komisaris,bagus
3,koruptor,reformasi,sangat
4,tapi,kesehatan,saya
5,aja,mas,bisa
6,yang,kelas prof,thumbs_up
7,itu,top,good
8,mereka,href url,baik
9,kalau,url,pasti


Kelas F1 terendah: neutral | F1: 0.56 | support: 89
Slice error tertinggi dengan n>=20:


,slice,nilai,n,error_%,accuracy,macro_f1_kelas_hadir
15,confidence_label,"(-inf, 0.6]",88,61.363636,0.386364,0.374039
16,confidence_label,"(0.6, 0.8]",146,40.410959,0.595890,0.578048
10,url,True,26,38.461538,0.615385,0.535714
0,panjang,"(-1.0, 5.0]",145,33.793103,0.662069,0.626199
17,confidence_label,"(0.8, 0.95]",206,29.126214,0.708738,0.689446


## 16 · Ringkasan keputusan & keluaran

File prediksi bernama holdout_predictions, bukan submission kompetisi. Tidak ada label holdout yang dipakai fitting. Untuk deployment/kompetisi, retrain setelah prosedur dipilih, lalu transform test tanpa fit ulang vocabulary pada test.

In [41]:
decisions=pd.DataFrame([
    {"temuan":"duplikat/near-duplicate","bukti":f"{data.dup_group.nunique()} grup / {len(data)} baris","tindakan":"grup dibentuk sebelum split; overlap=0"},
    {"temuan":"preprocessing","bukti":best_view,"tindakan":"dipilih lewat ablasi validation; BERT memakai resep sendiri"},
    {"temuan":"panjang token","bukti":f"{token_lengths.gt(MAX_LEN).mean():.2%} train > {MAX_LEN}","tindakan":"batas eksplisit; slice truncation dilaporkan"},
    {"temuan":"model pilihan","bukti":best_name,"tindakan":"dipilih validation, holdout hanya laporan"},
    {"temuan":"post-processing","bukti":str(use_post),"tindakan":"pengali hanya jika cross-fit validation naik > MIN_POST_GAIN"},
    {"temuan":"kelas terlemah","bukti":str(weakest.kelas),"tindakan":"baca error, label ambigu, support, dan konteks sebelum eksperimen berikutnya"}])
show("Temuan → keputusan",decisions)
leaderboard.to_csv(OUT/"validation_models.csv",index=False)
hold_table.to_csv(OUT/"holdout_models.csv")
per_class.to_csv(OUT/"per_class.csv",index=False);slices.to_csv(OUT/"error_slices.csv",index=False)
analysis[["row_id",TEXT_COL,LABEL_COL,"prediksi","salah","skor_model"]].to_csv(OUT/"holdout_predictions.csv",index=False)


Temuan → keputusan


,temuan,bukti,tindakan
0,duplikat/near-duplicate,4381 grup / 5000 baris,grup dibentuk sebelum split; overlap=0
1,preprocessing,normalized,dipilih lewat ablasi validation; BERT memakai resep sendiri
2,panjang token,6.04% train > 128,batas eksplisit; slice truncation dilaporkan
3,model pilihan,05_indobertweet,"dipilih validation, holdout hanya laporan"
4,post-processing,True,pengali hanya jika cross-fit validation naik > MIN_POST_GAIN
5,kelas terlemah,neutral,"baca error, label ambigu, support, dan konteks sebelum eksperimen berikutnya"


In [42]:
import importlib.metadata as metadata
manifest={"run":"multiclass_review","rows":len(data),"raw_rows":len(raw),"split_rows":{"train":len(train),"validation":len(val),"holdout":len(holdout)},
    "dataset_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),"seed":SEED,"models_evaluated":len(hold_pred),
    "selection":selection,"final_holdout":metrics(y_hold,final_pred),"gpu":torch.cuda.get_device_name(0),
    "gpu_peak_GB":torch.cuda.max_memory_allocated()/1e9,"seconds":time.perf_counter()-RUN_START,
    "versions":{k:metadata.version(k) for k in ["torch","transformers","scikit-learn","gensim","pandas"]}}
assert manifest["gpu_peak_GB"]>0
(OUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

{
  "run": "multiclass_review",
  "rows": 5000,
  "raw_rows": 5035,
  "split_rows": {
    "train": 2966,
    "validation": 1016,
    "holdout": 1018
  },
  "dataset_sha256": "a0283b93e120dd5c411bd37117c38879da21cdb39c6c7ec4c9fafd11d7956fba",
  "seed": 42,
  "models_evaluated": 5,
  "selection": {
    "best_validation_model": "05_indobertweet",
    "postprocessing": true,
    "multipliers": [
      1.5,
      1.0,
      0.5
    ],
    "classes": [
      "negative",
      "neutral",
      "positive"
    ]
  },
  "final_holdout": {
    "macro_f1": 0.707411024134294,
    "weighted_f1": 0.8111519309448098,
    "accuracy": 0.8045186640471512,
    "balanced_acc": 0.7420394718538869
  },
  "gpu": "Tesla T4",
  "gpu_peak_GB": 2.685613568,
  "seconds": 264.70508257299997,
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "scikit-learn": "1.6.1",
    "gensim": "4.4.0",
    "pandas": "2.3.3"
  }
}


## Sumber & batas interpretasi
- Fungsi preprocessing disalin dari notebook universal proyek ini; kamus contoh harus ditinjau untuk domain baru. Tidak ada jaminan normalisasi selalu menaikkan skor.
- [IndoBERTweet model card](https://huggingface.co/indolem/indobertweet-base-uncased): input lowercase, placeholder mention/URL, emoji ke teks.
- [scikit-learn: leakage](https://scikit-learn.org/stable/common_pitfalls.html), [group cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html): fit representasi hanya train, pisahkan grup terkait.
- [Gensim Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html): embedding corpus kecil tidak setara pretrained embedding korpus besar.
- Holdout tunggal memberi satu estimasi. Setelah pola notebook disetujui, repeated/group CV dan beberapa seed dapat mengukur variasi hasil. Hasil pseudo-label perlu dibedakan dari evaluasi label manusia.